In [ ]:
#+------------------------------------------------------------------+
#|  Minimal Grid Martingale — Python backtest                        |
#|  Entries at bar open. Hedging (each grid add is separate).        |
#+------------------------------------------------------------------+
from dataclasses import dataclass, field
from typing import List


#------------------------------------------------------------------
# Config
#------------------------------------------------------------------
@dataclass
class Config:
    first_lot:        float = 0.01
    distance_pips:    float = 55.0       # grid step in pips
    pip_size:         float = 0.0001     # price units per pip (EURUSD 5-digit)
    max_orders:       int   = 10
    tp_in_money:      float = 0.50       # close basket when floating profit >= this
    lot_multiplier:   float = 2.0
    contract_size:    float = 100_000.0  # 1 lot = 100k units (EURUSD)
    min_lot:          float = 0.01
    lot_step:         float = 0.01


#------------------------------------------------------------------
# Data structures
#------------------------------------------------------------------
@dataclass
class Position:
    side:      int          # +1 = long, -1 = short
    entry_idx: int          # bar index at fill
    entry_px:  float        # fill price (= open[entry_idx])
    lots:      float

    def unrealized(self, price: float, cfg: Config) -> float:
        # P/L in quote currency for a 1-lot = contract_size units position
        move = (price - self.entry_px) * self.side
        return move * self.lots * cfg.contract_size


@dataclass
class Trade:
    side:      int
    entry_idx: int
    entry_px:  float
    exit_idx:  int
    exit_px:   float
    lots:      float
    pnl:       float


@dataclass
class BacktestResult:
    trades:      List[Trade] = field(default_factory=list)
    equity:      List[float] = field(default_factory=list)
    balance:     float = 0.0
    max_dd:      float = 0.0


#------------------------------------------------------------------
# Core backtest loop
#------------------------------------------------------------------
def backtest(bars: List[dict], cfg: Config, starting_balance: float = 10_000.0) -> BacktestResult:
    """
    bars: list of dicts with keys 'open','high','low','close' (and optional 'time').
          Index 0 = oldest bar.

    Logic mirrors the MQL5 EA:
      - On each new bar i (i >= 2), look at close[i-1] vs close[i-2].
      - If no open positions: open first trade at open[i] in that direction.
      - If positions exist (same direction, no hedge):
          if open[i] moved >= distance_pips against last entry:
             add a position of last_lots * lot_multiplier at open[i].
      - Always, on every bar (not just entries): if sum(unrealized) >= tp_in_money:
             close all at open[i], realise P/L.
    """
    res = BacktestResult()
    balance = starting_balance
    positions: List[Position] = []
    res.equity.append(balance)

    step = cfg.distance_pips * cfg.pip_size

    n = len(bars)
    for i in range(2, n):
        o = bars[i]['open']

        #--- 1. EXIT check -----------------------------------------
        if positions:
            floating = sum(p.unrealized(o, cfg) for p in positions)
            if floating >= cfg.tp_in_money:
                for p in positions:
                    pnl = p.unrealized(o, cfg)
                    balance += pnl
                    res.trades.append(Trade(
                        side=p.side,
                        entry_idx=p.entry_idx, entry_px=p.entry_px,
                        exit_idx=i,            exit_px=o,
                        lots=p.lots,           pnl=pnl,
                    ))
                positions.clear()
                res.equity.append(balance)
                continue   # bar consumed by exit; wait for next bar

        #--- 2. ENTRY / GRID ADD -----------------------------------
        prev_close = bars[i-1]['close']
        curr_close = bars[i-2]['close']   # see note below

        # NOTE: signal uses the two bars just BEFORE the entry bar.
        # In MQL4/5 we used Close[2] (bar before last) vs Close[1] (last closed bar)
        # evaluated on the tick where bar i = 0 is opening.
        # Here we are AT bar i entering, so:
        #   "last closed bar"   = i-1
        #   "bar before that"   = i-2
        # so signal = close[i-1] vs close[i-2].
        signal_last   = bars[i-1]['close']
        signal_before = bars[i-2]['close']

        if not positions:
            # first trade of a fresh basket
            if signal_before < signal_last:      # bullish
                positions.append(Position(side=+1, entry_idx=i, entry_px=o,
                                          lots=cfg.first_lot))
            elif signal_before > signal_last:    # bearish
                positions.append(Position(side=-1, entry_idx=i, entry_px=o,
                                          lots=cfg.first_lot))
        else:
            # grid add (same direction only — no hedge)
            if len(positions) < cfg.max_orders:
                anchor = positions[-1]           # most recent position
                next_lot = _round_lot(anchor.lots * cfg.lot_multiplier, cfg)

                if anchor.side == +1:
                    # BUY basket: add when price drops one step below last entry
                    if o <= anchor.entry_px - step:
                        positions.append(Position(side=+1, entry_idx=i, entry_px=o,
                                                  lots=next_lot))
                else:
                    # SELL basket: add when price rises one step above last entry
                    if o >= anchor.entry_px + step:
                        positions.append(Position(side=-1, entry_idx=i, entry_px=o,
                                                  lots=next_lot))

        #--- 3. equity mark-to-market (for drawdown tracking) ------
        floating = sum(p.unrealized(o, cfg) for p in positions)
        eq = balance + floating
        res.equity.append(eq)
        if eq > starting_balance:
            peak = max(res.equity)
        # simple running max drawdown
        peak = max(res.equity) if res.equity else starting_balance
        dd = (peak - eq) / peak if peak > 0 else 0.0
        if dd > res.max_dd:
            res.max_dd = dd

    #--- force-close any remaining positions at last bar's close ---
    if positions:
        last_px = bars[-1]['close']
        for p in positions:
            pnl = p.unrealized(last_px, cfg)
            balance += pnl
            res.trades.append(Trade(
                side=p.side,
                entry_idx=p.entry_idx, entry_px=p.entry_px,
                exit_idx=len(bars)-1,  exit_px=last_px,
                lots=p.lots,           pnl=pnl,
            ))
        positions.clear()

    res.balance = balance
    return res


def _round_lot(lot: float, cfg: Config) -> float:
    steps = round(lot / cfg.lot_step)
    return max(cfg.min_lot, steps * cfg.lot_step)


#------------------------------------------------------------------
# Example driver
#------------------------------------------------------------------
if __name__ == "__main__":
    import random

    # generate fake 1-min bars (random walk) for a smoke test
    random.seed(1)
    px = 1.1000
    bars = []
    for _ in range(20_000):
        o = px
        h = o + random.uniform(0, 0.0003)
        l = o - random.uniform(0, 0.0003)
        c = random.uniform(l, h)
        bars.append({'open': o, 'high': h, 'low': l, 'close': c, 'volume': 1})
        px = c

    cfg = Config(
        first_lot      = 0.01,
        distance_pips  = 55,
        pip_size       = 0.0001,
        max_orders     = 10,
        tp_in_money    = 0.50,
        lot_multiplier = 2.0,
    )

    res = backtest(bars, cfg, starting_balance=10_000.0)

    print(f"trades     : {len(res.trades)}")
    print(f"final bal  : {res.balance:.2f}")
    print(f"return     : {100*(res.balance/10_000 - 1):.2f}%")
    print(f"max DD     : {100*res.max_dd:.2f}%")

    # show the largest basket
    from collections import defaultdict
    baskets = defaultdict(list)
    basket_id = 0
    last_exit = -1
    for t in res.trades:
        if t.entry_idx < last_exit:
            basket_id += 1
        baskets[basket_id].append(t)
        last_exit = t.exit_idx
    biggest = max(baskets.values(), key=len)
    print(f"\nbiggest basket: {len(biggest)} orders")
    for t in biggest:
        print(f"  side={t.side:+d} lots={t.lots:.2f} "
              f"entry@{t.entry_px:.5f} exit@{t.exit_px:.5f} pnl={t.pnl:+.2f}")